Organize by Temporal Phase + Merge (<30 min apart)

In [9]:
import os
import re
import glob
import shutil
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

In [10]:
# ================== User Settings ==================
INPUT_FOLDER = r"/Users/ks/Desktop/Wu/Testing"   # folder full of ECOSTRESS tiffs

MERGE_WINDOW_MIN = 30   # merge files less than this many minutes apart

FILENAME_PATTERN = re.compile(
    r'^\d+_\d+_(\d{8}T\d{6})_\d+_\d+\.tiff?$', re.IGNORECASE
)

PHASE_CATEGORIES = ["Night", "Morning", "Afternoon", "Evening"]

In [11]:
def parse_timestamp_from_name(path):
    """
    Extract UTC datetime from a filename like:
    34107_003_20240712T035700_0601_01.tif
    Returns a datetime, or None if the filename doesn't match.
    """
    filename = os.path.basename(path)
    match = FILENAME_PATTERN.match(filename)
    if not match:
        print(f"Skipping malformed filename: {filename}")
        return None
    try:
        return datetime.strptime(match.group(1), "%Y%m%dT%H%M%S").replace(
            tzinfo=ZoneInfo("UTC")
        )
    except ValueError:
        print(f"Could not parse timestamp in: {filename}")
        return None


def classify_time(hour):
    """Classify an hour (0-23, local time) into a temporal phase."""
    if 0 <= hour < 5:
        return "Night"
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    else:
        return "Evening"

In [12]:
def organize_tifs_by_phase(folder_path, local_tz="America/Los_Angeles"):
    """
    Moves each .tif/.tiff in folder_path into a Night/Morning/Afternoon/Evening
    subfolder based on its local (Pacific) capture time.
    Returns dict of {phase: [(timestamp_str, filepath), ...]}
    """
    for cat in PHASE_CATEGORIES:
        os.makedirs(os.path.join(folder_path, cat), exist_ok=True)

    classified = {cat: [] for cat in PHASE_CATEGORIES}

    for filename in os.listdir(folder_path):
        if not filename.lower().endswith((".tif", ".tiff")):
            continue

        source_path = os.path.join(folder_path, filename)
        dt_utc = parse_timestamp_from_name(source_path)
        if dt_utc is None:
            continue

        dt_local = dt_utc.astimezone(ZoneInfo(local_tz))
        phase = classify_time(dt_local.hour)

        dest_path = os.path.join(folder_path, phase, filename)
        shutil.move(source_path, dest_path)

        classified[phase].append((dt_local.strftime("%Y-%m-%d %H:%M:%S %Z"), dest_path))

    return classified

In [13]:
def group_by_time(files, max_minutes=30):
    """
    Chains files sorted by timestamp into groups where each file is within
    max_minutes of the PREVIOUS file in the group (so a chain of near files
    can span more than max_minutes total, but every consecutive gap is small).
    """
    max_delta = timedelta(minutes=max_minutes)

    items = [(parse_timestamp_from_name(f), f) for f in files]
    items = [(dt, f) for dt, f in items if dt is not None]
    items.sort(key=lambda x: x[0])

    groups = []
    current_group = []

    for dt, f in items:
        if not current_group:
            current_group = [(dt, f)]
            continue

        prev_dt = current_group[-1][0]
        if dt - prev_dt <= max_delta:
            current_group.append((dt, f))
        else:
            groups.append(current_group)
            current_group = [(dt, f)]

    if current_group:
        groups.append(current_group)

    return [[f for _, f in grp] for grp in groups]


def mosaic_mean(files, out_path):
    """Mosaic a list of GeoTIFFs using the mean for overlapping pixels."""
    if len(files) == 0:
        return

    datasets = [rasterio.open(f) for f in files]
    ref = datasets[0]
    ref_crs = ref.crs

    xres, yres = ref.res
    xres, yres = abs(xres), abs(yres)

    minxs, maxxs, minys, maxys = [], [], [], []
    for ds in datasets:
        if ds.crs != ref_crs:
            raise ValueError("All rasters must have the same CRS.")
        b = ds.bounds
        minxs.append(min(b.left, b.right))
        maxxs.append(max(b.left, b.right))
        minys.append(min(b.bottom, b.top))
        maxys.append(max(b.bottom, b.top))

    minx, maxx = min(minxs), max(maxxs)
    miny, maxy = min(minys), max(maxys)

    width = int(np.ceil((maxx - minx) / xres))
    height = int(np.ceil((maxy - miny) / yres))
    mosaic_transform = from_bounds(minx, miny, maxx, maxy, width, height)

    n = len(datasets)
    stack = np.full((n, height, width), np.nan, dtype="float32")

    for i, ds in enumerate(datasets):
        data = ds.read(1).astype("float32")
        if ds.nodata is not None and not np.isnan(ds.nodata):
            data = np.where(data == ds.nodata, np.nan, data)

        dest = np.full((height, width), np.nan, dtype="float32")
        reproject(
            source=data,
            destination=dest,
            src_transform=ds.transform,
            src_crs=ds.crs,
            dst_transform=mosaic_transform,
            dst_crs=ref_crs,
            src_nodata=None,
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )
        stack[i] = dest

    mosaic_arr = np.nanmean(stack, axis=0).astype("float32")

    meta = ref.meta.copy()
    meta.update({
        "height": height,
        "width": width,
        "transform": mosaic_transform,
        "dtype": "float32",
        "nodata": np.nan,
    })

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mosaic_arr, 1)

    for ds in datasets:
        ds.close()

In [14]:
def process_folder(input_folder, merge_window_min=MERGE_WINDOW_MIN):
    print(f"Organizing tiffs in {input_folder} by temporal phase...")
    classified = organize_tifs_by_phase(input_folder)

    for phase, entries in classified.items():
        print(f"\n{phase}: {len(entries)} file(s)")
        for ts, path in entries:
            print(f"  {ts}  -> {os.path.basename(path)}")

    for phase in PHASE_CATEGORIES:
        phase_folder = os.path.join(input_folder, phase)
        files = sorted(glob.glob(os.path.join(phase_folder, "*.tif*")))
        if not files:
            continue

        groups = group_by_time(files, max_minutes=merge_window_min)
        print(f"\n[{phase}] {len(groups)} merge group(s) from {len(files)} file(s)")

        merged_folder = os.path.join(phase_folder, "merged")

        for i, group_files in enumerate(groups, start=1):
            if len(group_files) == 1:
                # nothing to merge, leave as-is
                continue

            dts = [parse_timestamp_from_name(f) for f in group_files]
            first_dt, last_dt = min(dts), max(dts)
            ts1 = first_dt.strftime("%Y%m%dT%H%M%S")
            ts2 = last_dt.strftime("%Y%m%dT%H%M%S")
            out_name = f"{phase}_group{i:03d}_{ts1}_to_{ts2}_mean.tif"
            out_path = os.path.join(merged_folder, out_name)

            print(f"  Merging {len(group_files)} files -> {out_name}")
            for f in group_files:
                print(f"    {os.path.basename(f)}")

            mosaic_mean(group_files, out_path)

    print("\nDone.")

In [15]:
process_folder(INPUT_FOLDER, merge_window_min=MERGE_WINDOW_MIN)

Organizing tiffs in /Users/ks/Desktop/Wu/Testing by temporal phase...

Night: 11 file(s)
  2024-06-30 01:43:49 PDT  -> 33924_006_20240630T084349_0601_01.tif
  2024-07-04 00:07:59 PDT  -> 33985_001_20240704T070759_0601_01.tif
  2024-06-22 04:55:22 PDT  -> 33802_006_20240622T115522_0601_02.tif
  2024-07-19 01:03:22 PDT  -> 34218_010_20240719T080322_0601_01.tif
  2024-06-29 02:31:14 PDT  -> 33909_005_20240629T093114_0601_01.tif
  2024-07-16 01:51:33 PDT  -> 34172_010_20240716T085133_0601_01.tif
  2024-06-30 01:42:57 PDT  -> 33924_005_20240630T084257_0601_01.tif
  2024-06-26 03:19:00 PDT  -> 33863_005_20240626T101900_0601_02.tif
  2024-06-22 04:54:30 PDT  -> 33802_005_20240622T115430_0601_02.tif
  2024-06-23 04:06:25 PDT  -> 33817_003_20240623T110625_0601_02.tif
  2024-07-04 00:08:51 PDT  -> 33985_002_20240704T070851_0601_01.tif

Morning: 8 file(s)
  2024-07-07 05:51:02 PDT  -> 34035_011_20240707T125102_0601_01.tif
  2024-07-08 05:03:21 PDT  -> 34050_009_20240708T120321_0601_01.tif
  2024-

/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_23934/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")


  Merging 2 files -> Night_group005_20240630T084257_to_20240630T084349_mean.tif
    33924_005_20240630T084257_0601_01.tif
    33924_006_20240630T084349_0601_01.tif


/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_23934/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")


  Merging 2 files -> Night_group006_20240704T070759_to_20240704T070851_mean.tif
    33985_001_20240704T070759_0601_01.tif
    33985_002_20240704T070851_0601_01.tif


...


[Evening] 6 merge group(s) from 8 file(s)
  Merging 2 files -> Evening_group002_20240708T053240_to_20240708T053332_mean.tif
    34046_001_20240708T053240_0601_02.tif
    34046_002_20240708T053332_0601_02.tif


/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_23934/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")


  Merging 2 files -> Evening_group003_20240712T035700_to_20240712T035752_mean.tif
    34107_003_20240712T035700_0601_01.tif
    34107_004_20240712T035752_0601_01.tif


/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_23934/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")



Done.
